# NB-A - Inference et feedback Gradio

Cette application classe une image CIFAR-10 et collecte un feedback valide. Chaque lot de cinq nouvelles images cree une version publiee et incrementale du dataset ClearML.


In [ ]:
!pip -q install clearml gradio torch torchvision pillow python-dotenv


In [ ]:
# La configuration ClearML vient de variables d'environnement ou du fichier .env local.
# Ne mettez jamais de cles API dans le notebook.
import os
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
    env_file = find_dotenv(usecwd=True)
    if env_file:
        load_dotenv(env_file)
    else:
        load_dotenv()
except ImportError:
    pass

REQUIRED_CLEARML_VARS = (
    "CLEARML_WEB_HOST",
    "CLEARML_API_HOST",
    "CLEARML_FILES_HOST",
    "CLEARML_API_ACCESS_KEY",
    "CLEARML_API_SECRET_KEY",
)
missing = [name for name in REQUIRED_CLEARML_VARS if not os.getenv(name)]
if missing:
    raise RuntimeError(
        "Configuration ClearML absente. Renseignez .env ou les variables d'environnement : "
        + ", ".join(missing)
    )
print("Configuration ClearML chargee depuis l'environnement.")


In [ ]:
# NB-A - Inference Gradio et versioning incremental du feedback
import json
import os
import shutil
import time
import traceback
import uuid
from pathlib import Path
from typing import Tuple

import gradio as gr
import torch
import torch.nn as nn
from clearml import Dataset, Model
from PIL import Image
from torchvision import models, transforms

PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"
DATASET_NAME = "image-feedback"
BATCH_SIZE = 5
WORK_DIR = Path(os.getenv("MLOPS_WORK_DIR", Path.cwd() / "mlops_feedback"))
PENDING_DIR = WORK_DIR / "pending"
ARCHIVE_DIR = WORK_DIR / "published"
STATE_FILE = WORK_DIR / "state.json"
for directory in (PENDING_DIR, ARCHIVE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_resnet18(num_classes=10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_production_model() -> nn.Module:
    try:
        models_in_production = Model.query_models(
            tags=["production"], only_published=True, max_results=1
        ) or []
        if not models_in_production:
            print("Aucun modele de production : initialisation aleatoire temporaire.")
            model = build_resnet18().to(DEVICE)
            model.eval()
            return model
        registered_model = models_in_production[0]
        local_path = registered_model.get_local_copy()
        model = build_resnet18().to(DEVICE)
        model.load_state_dict(torch.load(local_path, map_location=DEVICE), strict=True)
        model.eval()
        print(f"Modele de production charge : {registered_model.id}")
        return model
    except Exception:
        print("Echec du chargement du modele de production :\n", traceback.format_exc())
        model = build_resnet18().to(DEVICE)
        model.eval()
        return model

model = load_production_model()

TFM = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_image(image: Image.Image) -> Tuple[str, float]:
    x = TFM(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    probabilities = torch.softmax(model(x), dim=1)[0]
    confidence, class_index = torch.max(probabilities, dim=0)
    return CLASSES[int(class_index)], float(confidence.item())

def _read_state():
    if not STATE_FILE.exists():
        return {"last_dataset_id": None}
    try:
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    except Exception:
        return {"last_dataset_id": None}

def _write_state(dataset_id: str):
    STATE_FILE.write_text(json.dumps({"last_dataset_id": dataset_id}), encoding="utf-8")

def _pending_count() -> int:
    return sum(1 for item in PENDING_DIR.rglob("*") if item.is_file())

def store_feedback(image: Image.Image, true_label: str) -> str:
    if true_label not in CLASSES:
        raise ValueError("Le feedback doit contenir une des 10 classes CIFAR-10.")
    target_dir = PENDING_DIR / true_label
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / f"{uuid.uuid4().hex}.png"
    image.convert("RGB").save(target_path, format="PNG")
    return str(target_path)

def publish_pending_feedback() -> tuple[bool, str]:
    pending_count = _pending_count()
    if pending_count < BATCH_SIZE:
        return False, f"{pending_count}/{BATCH_SIZE} feedbacks en attente"

    try:
        try:
            parent = Dataset.get(
                dataset_name=DATASET_NAME,
                dataset_project=PROJECT,
                dataset_tags=["auto-feedback"],
                only_completed=True,
            )
            parents = [parent]
        except Exception:
            parents = None

        dataset = Dataset.create(
            dataset_name=DATASET_NAME,
            dataset_project=PROJECT,
            dataset_tags=["auto-feedback"],
            parent_datasets=parents,
            description="Lot incremental de feedback valide pour le reentrainement.",
        )
        dataset.add_files(path=str(PENDING_DIR))
        dataset.upload()
        dataset.finalize()

        batch_archive = ARCHIVE_DIR / time.strftime("%Y%m%d-%H%M%S")
        PENDING_DIR.rename(batch_archive)
        PENDING_DIR.mkdir(parents=True, exist_ok=True)
        _write_state(dataset.id)
        print(f"Dataset publie : {dataset.id} ({pending_count} nouvelles images)")
        return True, dataset.id
    except Exception:
        print("Echec de publication du dataset :\n", traceback.format_exc())
        return False, "Erreur de publication : voir le journal du notebook."

def predict_and_store(image: Image.Image, store: bool, true_label: str):
    if image is None:
        return "error", 0.0, "Ajoutez une image avant de lancer la prediction."
    try:
        prediction, confidence = predict_image(image)
        status = f"Prediction : {prediction} ({confidence:.3f})"
        if store:
            saved_path = store_feedback(image, true_label)
            published, result = publish_pending_feedback()
            if published:
                status += f" | Nouvelle version ClearML publiee : {result}"
            else:
                status += f" | Feedback enregistre : {saved_path} | {result}"
        return prediction, confidence, status
    except Exception as exc:
        print("Erreur Gradio :\n", traceback.format_exc())
        return "error", 0.0, f"Erreur : {exc}"

with gr.Blocks(title="CIFAR-10 - prediction et feedback") as demo:
    gr.Markdown("## Prediction CIFAR-10 et collecte de feedback\nCochez l'enregistrement uniquement apres avoir choisi le vrai label.")
    with gr.Row():
        input_image = gr.Image(type="pil", label="Image a classifier")
        with gr.Column():
            save_feedback = gr.Checkbox(label="Enregistrer ce feedback", value=False)
            feedback_label = gr.Dropdown(choices=CLASSES, value=CLASSES[0], label="Vrai label")
            predict_button = gr.Button("Predire et enregistrer")
    with gr.Row():
        output_label = gr.Textbox(label="Classe predite")
        output_confidence = gr.Number(label="Confiance", precision=4)
    output_status = gr.Textbox(label="Statut")
    predict_button.click(
        predict_and_store,
        inputs=[input_image, save_feedback, feedback_label],
        outputs=[output_label, output_confidence, output_status],
    )

demo.launch(share=False)


███████████████████████████████ 100% | 42.73/42.73 MB [00:06<00:00, 10.48MB/s]: c:\Users\Ibrahim\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
██████████████████████████████ 100% | 42.73/42.73 MB [00:06<-00:00,  6.91MB/s]: 


Modele de production charge : ebd399f1e7fc4b0f87acb0698896dfaa
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


ClearML results page: https://app.clear.ml/projects/02b54b1bec194d95a616652f60df0792/tasks/d3a277942a9f4618b2aeafdecf67223d/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/02b54b1bec194d95a616652f60df0792/experiments/d3a277942a9f4618b2aeafdecf67223d
Generating SHA2 hash for 5 files


100%|██████████| 5/5 [00:00<00:00, 45789.34it/s]

Hash generation completed


File compression and upload completed: total size 0 B, 0 chunk(s) stored (average size 0 B)
Dataset publie : d3a277942a9f4618b2aeafdecf67223d (5 nouvelles images)
